# ***Verify the foreign-key relationships between fact and dimensions***

### 1) **Every asset_id in Fact_Stock_Prices should exist in Dim_Stock**

In [3]:
%%sql

SELECT distinct(p.stock_id), s.stock_id FROM Price.fact_stock_prices p left join Stock.dim_stock s 
on p.stock_id = s.stock_id
-- verified

StatementMeta(, 70e22e1a-aae6-4747-8bdd-4ff582b5239f, 4, Finished, Available, Finished, False)

<Spark SQL result set with 35 rows and 2 fields>

### 2) **Every date_id in Fact_Stock_Prices should exist in Dim_Date**

In [12]:
%%sql
SELECT p.date_id, d.date_id FROM Price.fact_stock_prices as p 
left Join Date.dim_date d
on p.date_id = d.date_id
WHERE d.date_id is null
-- verified (all rows matched)


StatementMeta(, 70e22e1a-aae6-4747-8bdd-4ff582b5239f, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [10]:
%%sql
SELECT COUNT(Date_id)  from Date.dim_date

StatementMeta(, 70e22e1a-aae6-4747-8bdd-4ff582b5239f, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

# ***Data Quality Check***

### **checking for <mark>Null</mark> value**

### 

In [1]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(date_id) AS null_date_id,
    COUNT(*) - COUNT(stock_id) AS null_stock_id,
    COUNT(*) - COUNT(open) AS null_open,
    COUNT(*) - COUNT(high) AS null_high,
    COUNT(*) - COUNT(low) AS null_low,
    COUNT(*) - COUNT(close) AS null_close,
    COUNT(*) - COUNT(volume) AS null_volume
FROM Price.fact_stock_prices;
-- all 0 so no null value

StatementMeta(, 04a0b32f-fca6-4675-91bf-aba2b83dc244, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 8 fields>

### **Checking for duplicates**

In [4]:
%%sql
SELECT date_id, stock_id, timestamp, COUNT(*) AS cnt
FROM Price.fact_stock_prices
GROUP BY date_id, stock_id, timestamp
HAVING COUNT(*) > 1;

StatementMeta(, 04a0b32f-fca6-4675-91bf-aba2b83dc244, 5, Finished, Available, Finished, False)

<Spark SQL result set with 25 rows and 4 fields>

In [5]:
%%sql
SELECT *
FROM Price.fact_stock_prices
WHERE date_id = '20260724'
  AND stock_id = 3
  AND timestamp = '2026-07-24T14:04:00Z';

StatementMeta(, 04a0b32f-fca6-4675-91bf-aba2b83dc244, 6, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 8 fields>

# **OHLC sanity check**

In [6]:
%%sql
SELECT *
FROM Price.fact_stock_prices
WHERE high < low
   OR high < open
   OR high < close
   OR low > open
   OR low > close;
-- verified

StatementMeta(, 04a0b32f-fca6-4675-91bf-aba2b83dc244, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 8 fields>

#